# Building an Image Classifier with CNNs

## Step 1: Introduction au CNN

### Qu'est-ce qu'un CNN ?
Un **réseau de neurones convolutif** (CNN) est un réseau de neurones conçu pour traiter des données en grille, typiquement des images. Au lieu de relier chaque pixel à chaque neurone (ce qui donnerait un nombre énorme de paramètres), un CNN fait glisser de petits **filtres** sur l'image pour détecter des motifs locaux (bords, textures, formes). Les premières couches détectent des motifs simples (contours), les couches profondes les combinent en motifs complexes (oreilles, museau, yeux).

Deux idées clés :
- **Connexions locales** : chaque neurone ne regarde qu'une petite région de l'image.
- **Partage des poids** : le même filtre est appliqué partout dans l'image, donc un motif est reconnu quelle que soit sa position, et le nombre de paramètres reste faible.

### Composants principaux

**Couche convolutive (`Conv2D`)** : applique $K$ filtres de taille $k \times k$ sur l'entrée. Chaque filtre produit une *carte de caractéristiques* (feature map). Nombre de paramètres d'une couche :
$$(k \times k \times C_{in} + 1) \times K$$
où $C_{in}$ est le nombre de canaux d'entrée et le $+1$ correspond au biais.

**Fonction d'activation (ReLU)** : $f(x) = \max(0, x)$. Elle introduit de la non-linéarité (sans elle, empiler des couches équivaudrait à une seule transformation linéaire). Elle est simple, rapide à calculer et limite le problème de disparition du gradient.

**Couche de pooling (`MaxPooling2D`)** : réduit la taille spatiale (ici de moitié avec une fenêtre 2×2) en gardant le maximum de chaque fenêtre. Cela diminue le nombre de calculs, apporte une petite invariance aux translations et réduit le risque de surapprentissage. Elle n'a **aucun paramètre**.

**Couche entièrement connectée (`Dense`)** : après `Flatten`, chaque neurone est relié à toutes les entrées. Elle combine les caractéristiques extraites pour décider. Paramètres : $(n_{in} + 1) \times n_{out}$.

**Sortie sigmoïde** : pour une classification binaire (chat / chien), un seul neurone avec activation sigmoïde $\sigma(x) = \frac{1}{1+e^{-x}}$ donne une probabilité entre 0 et 1.

## Configuration

In [1]:
import os, random, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("GPU détecté :", tf.config.list_physical_devices("GPU"))

TensorFlow : 2.22.0-rc0
GPU détecté : []


## Étape 2 : Exploration du jeu de données

On utilise le jeu `cats_and_dogs_filtered` (sous-ensemble du dataset Kaggle *Dogs vs Cats*), qui contient déjà un dossier `train` et un dossier `validation`.

### Load the dataset

In [2]:
import os

# Chemin relatif vers le dossier
BASE_DIR = "cats_and_dogs_filtered"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "validation")

# Vérification de l'existence des répertoires
assert os.path.isdir(TRAIN_DIR) and os.path.isdir(VAL_DIR), "Vérifiez la structure du dossier cats_and_dogs_filtered"
print("Classes dans train :", sorted(os.listdir(TRAIN_DIR)))

Classes dans train : ['cats', 'dogs']


### Combien d'images de chats et de chiens y a-t-il ?

In [3]:
def count_images(directory):
    counts = {}
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([f for f in os.listdir(cls_path)
                               if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)

df_counts = pd.DataFrame({"train": train_counts, "validation": val_counts})
df_counts.loc["total"] = df_counts.sum()
print(df_counts)

       train  validation
cats    1000         500
dogs    1000         500
total   2000        1000
